# England & Wales Neighbourhood Cost-Function Finder

Scores every **LSOA** (~35 000 neighbourhoods, ≈ 1 500 residents each) against six
desirability criteria and ranks them.

| # | Criterion | Type |
|---|-----------|------|
| 1 | Close to an Area of Outstanding Natural Beauty | Distance (closer = better) |
| 2 | Less than 30 minutes from the sea | Distance (≤ 25 km straight-line proxy) |
| 3 | Constituency voted Labour / Green / Lib Dem (2024) | Binary filter |
| 4 | Low density of vape & kebab shops | Count (fewer = better) |
| 5 | Less than 1 mile from a train station | Distance (closer = better) |
| 6 | Far from a university | Distance (further = better) |

**Outputs:** interactive Folium map, static matplotlib map, ranked CSV.

## 1 — Imports & configuration

In [ ]:
import json
import time
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy.spatial import cKDTree
from shapely.geometry import Point, shape
from shapely.ops import unary_union
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

# ── paths ──
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# ── scoring parameters (easily tuneable) ──
WEIGHTS = {
    "aonb": 1.0,
    "sea": 1.0,
    "constituency": 1.0,   # binary: 1 if qualifying, else excluded
    "vape_kebab": 1.0,
    "train": 1.0,
    "university": 1.0,
}

# ── British National Grid for accurate distances ──
BNG = "EPSG:27700"
WGS84 = "EPSG:4326"

print("Setup complete.")

## 2 — Download helpers

In [ ]:
def fetch_arcgis_features(base_url, where="1=1", out_fields="*",
                          batch_size=2000, max_features=None):
    """Page through an ArcGIS REST FeatureServer and return a GeoDataFrame."""
    all_features = []
    offset = 0
    while True:
        params = {
            "where": where,
            "outFields": out_fields,
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": batch_size,
            "returnGeometry": "true",
        }
        resp = requests.get(f"{base_url}/query", params=params, timeout=120)
        resp.raise_for_status()
        data = resp.json()
        features = data.get("features", [])
        if not features:
            break
        all_features.extend(features)
        offset += len(features)
        if max_features and offset >= max_features:
            break
        # ArcGIS signals no more data when fewer results than requested
        if len(features) < batch_size:
            break
        time.sleep(0.25)  # be polite
    geojson = {"type": "FeatureCollection", "features": all_features}
    return gpd.GeoDataFrame.from_features(geojson, crs=WGS84)


def overpass_query(query_body, cache_file=None):
    """Run an Overpass API query and return a GeoDataFrame of points."""
    if cache_file and Path(cache_file).exists():
        return gpd.read_file(cache_file)
    full_query = f"[out:json][timeout:120];{query_body}out center;"
    resp = requests.post(
        "https://overpass-api.de/api/interpreter",
        data={"data": full_query}, timeout=180,
    )
    resp.raise_for_status()
    elements = resp.json().get("elements", [])
    rows = []
    for el in elements:
        lat = el.get("lat") or el.get("center", {}).get("lat")
        lon = el.get("lon") or el.get("center", {}).get("lon")
        if lat and lon:
            rows.append({"geometry": Point(lon, lat),
                         "name": el.get("tags", {}).get("name", ""),
                         "osm_id": el.get("id")})
    gdf = gpd.GeoDataFrame(rows, crs=WGS84)
    if cache_file:
        gdf.to_file(cache_file, driver="GeoJSON")
    return gdf


def cached_download(url, dest_path, **kwargs):
    """Download a file if it doesn't already exist locally."""
    dest_path = Path(dest_path)
    if dest_path.exists():
        print(f"  [cached] {dest_path.name}")
        return dest_path
    print(f"  Downloading {dest_path.name} ...")
    resp = requests.get(url, timeout=300, **kwargs)
    resp.raise_for_status()
    dest_path.write_bytes(resp.content)
    return dest_path


print("Helpers defined.")

---
## Phase 1 — Data acquisition

Each dataset is downloaded once and cached to `data/`.

### 3 — LSOA population-weighted centroids

In [ ]:
LSOA_CENTROIDS_CACHE = DATA_DIR / "lsoa_centroids.geojson"

if LSOA_CENTROIDS_CACHE.exists():
    print("Loading cached LSOA centroids ...")
    lsoa_centroids = gpd.read_file(LSOA_CENTROIDS_CACHE)
else:
    print("Downloading LSOA 2021 population-weighted centroids from ONS ...")
    # ONS ArcGIS FeatureServer for LSOA (Dec 2021) PWC
    LSOA_PWC_URL = (
        "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA_Dec_2021_PWC_for_England_and_Wales_2022_V3/FeatureServer/0"
    )
    lsoa_centroids = fetch_arcgis_features(LSOA_PWC_URL)
    lsoa_centroids.to_file(LSOA_CENTROIDS_CACHE, driver="GeoJSON")

print(f"LSOA centroids: {len(lsoa_centroids):,} rows")
print(lsoa_centroids.columns.tolist())
lsoa_centroids.head(3)

### 4 — LSOA boundaries (generalised, for map display)

In [ ]:
LSOA_BOUNDARIES_CACHE = DATA_DIR / "lsoa_boundaries.geojson"

if LSOA_BOUNDARIES_CACHE.exists():
    print("Loading cached LSOA boundaries ...")
    lsoa_boundaries = gpd.read_file(LSOA_BOUNDARIES_CACHE)
else:
    print("Downloading LSOA 2021 boundaries (BGC / super-generalised) from ONS ...")
    # Super-generalised clipped boundaries — much smaller file, fine for choropleth
    LSOA_BGC_URL = (
        "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA_Dec_2021_Boundaries_EW_BSC_V4/FeatureServer/0"
    )
    lsoa_boundaries = fetch_arcgis_features(LSOA_BGC_URL)
    lsoa_boundaries.to_file(LSOA_BOUNDARIES_CACHE, driver="GeoJSON")

print(f"LSOA boundaries: {len(lsoa_boundaries):,} rows")
lsoa_boundaries.head(3)

### 5 — Areas of Outstanding Natural Beauty (AONB / National Landscapes)

In [ ]:
AONB_CACHE = DATA_DIR / "aonb.geojson"

if AONB_CACHE.exists():
    print("Loading cached AONB polygons ...")
    aonb = gpd.read_file(AONB_CACHE)
else:
    print("Downloading AONB polygons ...")
    # England — Natural England open data
    AONB_ENG_URL = (
        "https://services.arcgis.com/JJzESW51TqeY9uj9/arcgis/rest/services/"
        "Areas_of_Outstanding_Natural_Beauty_England/FeatureServer/0"
    )
    aonb_eng = fetch_arcgis_features(AONB_ENG_URL)
    print(f"  England AONB: {len(aonb_eng)} features")

    # Wales — Natural Resources Wales
    AONB_WAL_URL = (
        "https://services1.arcgis.com/RRW3v3VFOfjJCYFq/arcgis/rest/services/"
        "NRW_AONB/FeatureServer/0"
    )
    try:
        aonb_wal = fetch_arcgis_features(AONB_WAL_URL)
        print(f"  Wales AONB: {len(aonb_wal)} features")
        aonb = pd.concat([aonb_eng, aonb_wal], ignore_index=True)
    except Exception as exc:
        print(f"  Wales AONB download failed ({exc}), using England only.")
        aonb = aonb_eng

    aonb = gpd.GeoDataFrame(aonb, crs=WGS84)
    aonb.to_file(AONB_CACHE, driver="GeoJSON")

print(f"AONB polygons: {len(aonb)} features")
aonb.head(3)

### 6 — Coastline (for sea-distance scoring)

In [ ]:
COAST_CACHE = DATA_DIR / "coastline_ew.geojson"

if COAST_CACHE.exists():
    print("Loading cached coastline ...")
    coastline = gpd.read_file(COAST_CACHE)
else:
    print("Downloading Natural Earth 1:10m coastline ...")
    coast_url = (
        "https://naciscdn.org/naturalearth/10m/physical/"
        "ne_10m_coastline.zip"
    )
    coast_zip = cached_download(coast_url, DATA_DIR / "ne_10m_coastline.zip")
    coastline_world = gpd.read_file(f"zip://{coast_zip}")

    # Clip to England & Wales bounding box (with generous buffer)
    from shapely.geometry import box
    ew_bbox = box(-6.5, 49.5, 2.5, 56.0)
    coastline = coastline_world.clip(ew_bbox)
    coastline = coastline[~coastline.is_empty].reset_index(drop=True)
    coastline.to_file(COAST_CACHE, driver="GeoJSON")

print(f"Coastline segments: {len(coastline)}")
coastline.head(3)

### 7 — Parliamentary constituencies & 2024 election results

In [ ]:
# ── 7a: LSOA-to-constituency lookup ──
LSOA_PCON_CACHE = DATA_DIR / "lsoa_pcon_lookup.csv"

if LSOA_PCON_CACHE.exists():
    print("Loading cached LSOA → PCON lookup ...")
    lsoa_pcon = pd.read_csv(LSOA_PCON_CACHE)
else:
    print("Downloading LSOA → PCON (2024) lookup from ONS ...")
    # ONS publishes LSOA (2021) to PCON (2024) best-fit lookup
    LSOA_PCON_URL = (
        "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA21_PCON25_LAD24_EW_LU/FeatureServer/0"
    )
    lsoa_pcon_gdf = fetch_arcgis_features(LSOA_PCON_URL, out_fields="*")
    lsoa_pcon = pd.DataFrame(lsoa_pcon_gdf.drop(columns="geometry", errors="ignore"))
    lsoa_pcon.to_csv(LSOA_PCON_CACHE, index=False)

print(f"LSOA→PCON lookup: {len(lsoa_pcon):,} rows")
print("Columns:", lsoa_pcon.columns.tolist())
lsoa_pcon.head(3)

In [ ]:
# ── 7b: 2024 General Election results ──
ELECTION_CACHE = DATA_DIR / "election_2024.csv"

if ELECTION_CACHE.exists():
    print("Loading cached election results ...")
    election = pd.read_csv(ELECTION_CACHE)
else:
    print("Downloading 2024 election results from Electoral Commission ...")
    # House of Commons Library publishes results as CSV
    ELECTION_URL = (
        "https://researchbriefings.files.parliament.uk/documents/CBP-10009/"
        "HoC-GE2024-results-by-constituency.csv"
    )
    resp = requests.get(ELECTION_URL, timeout=60)
    resp.raise_for_status()
    ELECTION_CACHE.write_bytes(resp.content)
    election = pd.read_csv(ELECTION_CACHE)

print(f"Election results: {len(election)} constituencies")
print("Columns:", election.columns.tolist())
election.head(3)

In [ ]:
# ── 7c: Identify qualifying constituencies ──
# Inspect column names and adapt — common names: 'first_party', 'Winner', etc.
print("Election columns:", election.columns.tolist())
print("\nSample rows:")
election.head()

In [ ]:
# Build the qualifying set — adapt column names after inspecting above
# Common column names in the HoC data:
#   'ons_id' or 'constituency_id' — ONS constituency code
#   'first_party' — winning party name

# Auto-detect the winning-party column
_party_col_candidates = [c for c in election.columns
                         if "party" in c.lower() or "winner" in c.lower()]
print("Likely party columns:", _party_col_candidates)

_id_col_candidates = [c for c in election.columns
                      if "ons" in c.lower() or "pcon" in c.lower()
                      or "constituency_id" in c.lower() or c == "id"]
print("Likely ID columns:", _id_col_candidates)

# ---- Set these after inspecting the output above ----
PARTY_COL = _party_col_candidates[0] if _party_col_candidates else "first_party"
PCON_ID_COL = _id_col_candidates[0] if _id_col_candidates else "ons_id"

QUALIFYING_PARTIES = {"lab", "labour", "green", "ld", "lib dem",
                      "liberal democrats", "liberal democrat",
                      "plaid cymru"}  # added Plaid for Welsh representation

election["qualifying"] = (
    election[PARTY_COL]
    .str.strip().str.lower()
    .isin(QUALIFYING_PARTIES)
)

qualifying_pcon_codes = set(
    election.loc[election["qualifying"], PCON_ID_COL].dropna()
)

print(f"\nQualifying constituencies: {len(qualifying_pcon_codes)} / {len(election)}")
print("Party distribution:")
print(election[PARTY_COL].value_counts())

### 8 — Train stations (NaPTAN)

In [ ]:
STATIONS_CACHE = DATA_DIR / "train_stations.geojson"

if STATIONS_CACHE.exists():
    print("Loading cached train stations ...")
    stations = gpd.read_file(STATIONS_CACHE)
else:
    print("Downloading NaPTAN rail stations ...")
    NAPTAN_CSV_URL = "https://beta-naptan.dft.gov.uk/Download/National/csv"
    naptan_path = cached_download(NAPTAN_CSV_URL, DATA_DIR / "naptan_national.csv")
    naptan = pd.read_csv(naptan_path, low_memory=False)
    print(f"  NaPTAN total stops: {len(naptan):,}")

    # Filter to railway stations only
    rail = naptan[naptan["StopType"].isin(["RLY", "RPL", "RSE"])].copy()
    print(f"  Rail stations: {len(rail):,}")

    # Convert to GeoDataFrame
    stations = gpd.GeoDataFrame(
        rail,
        geometry=gpd.points_from_xy(rail["Longitude"], rail["Latitude"]),
        crs=WGS84,
    )
    stations.to_file(STATIONS_CACHE, driver="GeoJSON")

print(f"Train stations: {len(stations):,}")
stations.head(3)

### 9 — Universities (from OpenStreetMap)

In [ ]:
UNIS_CACHE = DATA_DIR / "universities.geojson"

if UNIS_CACHE.exists():
    print("Loading cached universities ...")
    universities = gpd.read_file(UNIS_CACHE)
else:
    print("Querying Overpass API for universities in England & Wales ...")
    uni_query = """
    (
      nwr["amenity"="university"](49.5,-6.5,56.0,2.5);
    );
    """
    universities = overpass_query(uni_query, cache_file=str(UNIS_CACHE))

print(f"Universities: {len(universities)}")
universities.head(3)

### 10 — Vape shops & kebab shops (from OpenStreetMap)

In [ ]:
VAPE_CACHE = DATA_DIR / "vape_shops.geojson"
KEBAB_CACHE = DATA_DIR / "kebab_shops.geojson"

if VAPE_CACHE.exists():
    vape_shops = gpd.read_file(VAPE_CACHE)
    print(f"[cached] Vape shops: {len(vape_shops)}")
else:
    print("Querying Overpass API for vape shops ...")
    vape_query = """
    (
      nwr["shop"="e-cigarette"](49.5,-6.5,56.0,2.5);
      nwr["name"~"[Vv]ape"](49.5,-6.5,56.0,2.5);
    );
    """
    vape_shops = overpass_query(vape_query, cache_file=str(VAPE_CACHE))
    print(f"Vape shops: {len(vape_shops)}")

# Small delay to avoid Overpass rate-limiting
time.sleep(5)

if KEBAB_CACHE.exists():
    kebab_shops = gpd.read_file(KEBAB_CACHE)
    print(f"[cached] Kebab shops: {len(kebab_shops)}")
else:
    print("Querying Overpass API for kebab shops ...")
    kebab_query = """
    (
      nwr["cuisine"="kebab"](49.5,-6.5,56.0,2.5);
      nwr["name"~"[Kk]ebab"](49.5,-6.5,56.0,2.5);
    );
    """
    kebab_shops = overpass_query(kebab_query, cache_file=str(KEBAB_CACHE))
    print(f"Kebab shops: {len(kebab_shops)}")

print(f"\nTotal undesirable shops: {len(vape_shops) + len(kebab_shops)}")

---
## Phase 2 — Spatial processing & scoring

All distance calculations in **EPSG:27700** (British National Grid, metres).

### 11 — Prepare LSOA centroids in BNG

In [ ]:
# Standardise LSOA code column name
lsoa_code_col = [c for c in lsoa_centroids.columns if "LSOA" in c and "CD" in c]
lsoa_name_col = [c for c in lsoa_centroids.columns if "LSOA" in c and "NM" in c]
print("Code column:", lsoa_code_col)
print("Name column:", lsoa_name_col)

LSOA_CD = lsoa_code_col[0] if lsoa_code_col else lsoa_centroids.columns[0]
LSOA_NM = lsoa_name_col[0] if lsoa_name_col else lsoa_centroids.columns[1]

# Project to BNG
centroids_bng = lsoa_centroids.to_crs(BNG).copy()
centroids_bng["x"] = centroids_bng.geometry.x
centroids_bng["y"] = centroids_bng.geometry.y

# Build a scores DataFrame keyed on LSOA code
scores = centroids_bng[[LSOA_CD, LSOA_NM, "x", "y"]].copy()
scores = scores.rename(columns={LSOA_CD: "lsoa_code", LSOA_NM: "lsoa_name"})
print(f"\nScoring {len(scores):,} LSOAs")
scores.head()

### 12 — Criterion 1: AONB proximity

In [ ]:
print("Computing AONB proximity scores ...")
aonb_bng = aonb.to_crs(BNG)
aonb_union = unary_union(aonb_bng.geometry)

centroid_points = centroids_bng.geometry
aonb_distance_m = centroid_points.distance(aonb_union)

# Score: 1 if inside/touching AONB, tapers linearly to 0 at 30 km
AONB_MAX_DIST = 30_000  # metres
scores["score_aonb"] = np.clip(1 - aonb_distance_m / AONB_MAX_DIST, 0, 1)

print(f"  Mean: {scores['score_aonb'].mean():.3f}")
print(f"  LSOAs inside AONB: {(scores['score_aonb'] == 1.0).sum():,}")
scores["score_aonb"].describe()

### 13 — Criterion 2: Sea proximity

In [ ]:
print("Computing sea proximity scores ...")
coastline_bng = coastline.to_crs(BNG)

# Extract dense points along the coastline for KDTree (faster than geometry distance)
coast_coords = []
for geom in coastline_bng.geometry:
    if geom.geom_type == "MultiLineString":
        for line in geom.geoms:
            coast_coords.extend(line.coords)
    elif geom.geom_type == "LineString":
        coast_coords.extend(geom.coords)
coast_coords = np.array(coast_coords)
print(f"  Coastline points: {len(coast_coords):,}")

coast_tree = cKDTree(coast_coords)
centroid_xy = np.column_stack([scores["x"].values, scores["y"].values])
sea_dist_m, _ = coast_tree.query(centroid_xy)

# Score: 1 if ≤ 25 km (~30 min drive proxy), tapers to 0 at 50 km
SEA_CLOSE = 25_000
SEA_FAR = 50_000
scores["score_sea"] = np.clip(
    1 - (sea_dist_m - SEA_CLOSE) / (SEA_FAR - SEA_CLOSE), 0, 1
)

print(f"  Mean: {scores['score_sea'].mean():.3f}")
print(f"  LSOAs within 25 km of sea: {(scores['score_sea'] == 1.0).sum():,}")
scores["score_sea"].describe()

### 14 — Criterion 3: Constituency filter (binary)

In [ ]:
print("Applying constituency filter ...")

# Find the PCON code column in the lookup
pcon_code_cols = [c for c in lsoa_pcon.columns if "PCON" in c.upper() and "CD" in c.upper()]
lsoa_code_cols_lookup = [c for c in lsoa_pcon.columns if "LSOA" in c.upper() and "CD" in c.upper()]
print("  PCON code column:", pcon_code_cols)
print("  LSOA code column in lookup:", lsoa_code_cols_lookup)

LOOKUP_LSOA_CD = lsoa_code_cols_lookup[0] if lsoa_code_cols_lookup else lsoa_pcon.columns[0]
LOOKUP_PCON_CD = pcon_code_cols[0] if pcon_code_cols else lsoa_pcon.columns[1]

# Map each LSOA to its constituency code
lsoa_to_pcon = dict(zip(lsoa_pcon[LOOKUP_LSOA_CD], lsoa_pcon[LOOKUP_PCON_CD]))
scores["pcon_code"] = scores["lsoa_code"].map(lsoa_to_pcon)

# Binary score: 1 if qualifying, 0 if not
scores["score_constituency"] = scores["pcon_code"].isin(qualifying_pcon_codes).astype(float)

n_pass = (scores["score_constituency"] == 1.0).sum()
n_fail = (scores["score_constituency"] == 0.0).sum()
print(f"  Qualifying LSOAs: {n_pass:,} ({100*n_pass/len(scores):.1f}%)")
print(f"  Excluded LSOAs:   {n_fail:,} ({100*n_fail/len(scores):.1f}%)")

### 15 — Criterion 4: Vape / kebab shop density

In [ ]:
print("Computing vape/kebab density scores ...")

# Combine vape + kebab shops into one GeoDataFrame
undesirable_shops = pd.concat([vape_shops, kebab_shops], ignore_index=True)
undesirable_shops = gpd.GeoDataFrame(undesirable_shops, crs=WGS84)
print(f"  Total undesirable shops: {len(undesirable_shops):,}")

# Spatial join with LSOA boundaries to count shops per LSOA
lsoa_bounds = lsoa_boundaries.copy()
lsoa_bd_code_col = [c for c in lsoa_bounds.columns if "LSOA" in c and "CD" in c][0]
lsoa_bounds = lsoa_bounds.rename(columns={lsoa_bd_code_col: "lsoa_code"})

shops_in_lsoa = gpd.sjoin(
    undesirable_shops.to_crs(WGS84),
    lsoa_bounds[["lsoa_code", "geometry"]].to_crs(WGS84),
    how="inner", predicate="within"
)
shop_counts = shops_in_lsoa.groupby("lsoa_code").size().rename("shop_count")

scores = scores.merge(shop_counts, left_on="lsoa_code", right_index=True, how="left")
scores["shop_count"] = scores["shop_count"].fillna(0).astype(int)

# Score: fewer shops = higher score. Use percentile rank (inverted).
# LSOAs with 0 shops get score 1.0
max_shops = scores["shop_count"].quantile(0.95)  # Cap at 95th percentile
max_shops = max(max_shops, 1)  # avoid division by zero
scores["score_vape_kebab"] = np.clip(1 - scores["shop_count"] / max_shops, 0, 1)

print(f"  Shops per LSOA — mean: {scores['shop_count'].mean():.2f}, "
      f"max: {scores['shop_count'].max()}")
print(f"  LSOAs with 0 shops: {(scores['shop_count'] == 0).sum():,}")
scores["score_vape_kebab"].describe()

### 16 — Criterion 5: Train station proximity

In [ ]:
print("Computing train station proximity scores ...")
stations_bng = stations.to_crs(BNG)
station_xy = np.column_stack([stations_bng.geometry.x, stations_bng.geometry.y])

station_tree = cKDTree(station_xy)
train_dist_m, _ = station_tree.query(centroid_xy)

# Score: 1 if ≤ 1 mile (1609 m), tapers to 0 at 5 miles (8045 m)
TRAIN_CLOSE = 1_609   # 1 mile
TRAIN_FAR = 8_045     # 5 miles
scores["score_train"] = np.clip(
    1 - (train_dist_m - TRAIN_CLOSE) / (TRAIN_FAR - TRAIN_CLOSE), 0, 1
)

print(f"  Mean: {scores['score_train'].mean():.3f}")
print(f"  LSOAs within 1 mile: {(scores['score_train'] == 1.0).sum():,}")
scores["score_train"].describe()

### 17 — Criterion 6: University distance

In [ ]:
print("Computing university distance scores ...")
unis_bng = universities.to_crs(BNG)
uni_xy = np.column_stack([unis_bng.geometry.x, unis_bng.geometry.y])

uni_tree = cKDTree(uni_xy)
uni_dist_m, _ = uni_tree.query(centroid_xy)

# Score: further from university = better. 0 at 0 km, 1 at 20 km+
UNI_MAX_DIST = 20_000  # 20 km
scores["score_university"] = np.clip(uni_dist_m / UNI_MAX_DIST, 0, 1)

print(f"  Mean: {scores['score_university'].mean():.3f}")
print(f"  LSOAs > 20 km from uni: {(scores['score_university'] == 1.0).sum():,}")
scores["score_university"].describe()

---
## Phase 3 — Combine scores & rank

In [ ]:
score_cols = ["score_aonb", "score_sea", "score_constituency",
              "score_vape_kebab", "score_train", "score_university"]
weight_vals = [WEIGHTS[k] for k in ["aonb", "sea", "constituency",
                                     "vape_kebab", "train", "university"]]

# Weighted total score
scores["total_score"] = sum(
    scores[col] * w for col, w in zip(score_cols, weight_vals)
)

# Filter out non-qualifying constituencies (hard filter)
qualified = scores[scores["score_constituency"] == 1.0].copy()
qualified = qualified.sort_values("total_score", ascending=False).reset_index(drop=True)
qualified["rank"] = range(1, len(qualified) + 1)

print(f"Qualified LSOAs: {len(qualified):,} / {len(scores):,}")
print(f"Score range: {qualified['total_score'].min():.2f} – {qualified['total_score'].max():.2f}")
print(f"\nTop 20 neighbourhoods:")
qualified[["rank", "lsoa_name", "total_score"] + score_cols].head(20)

In [ ]:
# Export full ranked CSV
csv_path = RESULTS_DIR / "top_lsoas.csv"
qualified.drop(columns=["x", "y"], errors="ignore").to_csv(csv_path, index=False)
print(f"Saved ranked CSV to {csv_path} ({len(qualified):,} rows)")

---
## Phase 4 — Visualisation

### 18 — Static matplotlib map

In [ ]:
print("Building static map ...")

# Merge scores onto boundaries
lsoa_bd_for_map = lsoa_bounds.merge(
    scores[["lsoa_code", "total_score", "score_constituency"]],
    on="lsoa_code", how="left"
)
lsoa_bd_for_map = lsoa_bd_for_map.to_crs(BNG)

# Only plot qualifying LSOAs with colour; grey out excluded ones
qualifying_mask = lsoa_bd_for_map["score_constituency"] == 1.0

fig, ax = plt.subplots(1, 1, figsize=(10, 14))

# Plot excluded LSOAs in light grey
lsoa_bd_for_map[~qualifying_mask].plot(
    ax=ax, color="#e0e0e0", edgecolor="none"
)

# Plot qualifying LSOAs with score colormap
lsoa_bd_for_map[qualifying_mask].plot(
    ax=ax, column="total_score", cmap="RdYlGn",
    edgecolor="none", legend=True,
    legend_kwds={"label": "Desirability Score", "shrink": 0.6}
)

# Overlay AONB outlines
aonb_bng.boundary.plot(ax=ax, color="forestgreen", linewidth=0.5, alpha=0.6)

ax.set_title("England & Wales — Neighbourhood Desirability Score\n"
             "(grey = excluded constituencies)", fontsize=14)
ax.set_axis_off()
plt.tight_layout()

static_map_path = RESULTS_DIR / "desirability_map.png"
fig.savefig(static_map_path, dpi=200, bbox_inches="tight")
print(f"Saved static map to {static_map_path}")
plt.show()

### 19 — Interactive Folium map

In [ ]:
import folium
from branca.colormap import LinearColormap

print("Building interactive Folium map (this may take a minute) ...")

# Use only qualifying LSOAs for the choropleth to keep file size manageable
map_gdf = lsoa_bounds.merge(
    qualified[["lsoa_code", "lsoa_name", "total_score",
               "score_aonb", "score_sea", "score_vape_kebab",
               "score_train", "score_university", "rank"]],
    on="lsoa_code", how="inner"
).to_crs(WGS84)

# Simplify geometries to reduce file size
map_gdf["geometry"] = map_gdf.geometry.simplify(0.001, preserve_topology=True)

print(f"  Mapping {len(map_gdf):,} qualifying LSOAs")

# Centre on England
centre = [52.5, -1.5]
m = folium.Map(location=centre, zoom_start=7, tiles="cartodbpositron")

# Colour scale
vmin = map_gdf["total_score"].quantile(0.05)
vmax = map_gdf["total_score"].quantile(0.95)
colormap = LinearColormap(
    ["#d73027", "#fee08b", "#1a9850"],
    vmin=vmin, vmax=vmax,
    caption="Desirability Score"
)

# Add choropleth layer
def style_function(feature):
    score = feature["properties"].get("total_score", 0)
    return {
        "fillColor": colormap(score),
        "color": "#333333",
        "weight": 0.2,
        "fillOpacity": 0.7,
    }

tooltip = folium.GeoJsonTooltip(
    fields=["lsoa_name", "rank", "total_score",
            "score_aonb", "score_sea",
            "score_vape_kebab", "score_train", "score_university"],
    aliases=["Neighbourhood", "Rank", "Total Score",
             "AONB", "Sea", "Vape/Kebab", "Train", "University"],
    localize=True,
)

folium.GeoJson(
    map_gdf.to_json(),
    style_function=style_function,
    tooltip=tooltip,
).add_to(m)

colormap.add_to(m)

html_path = RESULTS_DIR / "lsoa_scores_map.html"
m.save(str(html_path))
print(f"Saved interactive map to {html_path}")
print(f"  File size: {html_path.stat().st_size / 1e6:.1f} MB")
m

### 20 — Score distributions & summary

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

plot_cols = ["score_aonb", "score_sea", "score_vape_kebab",
             "score_train", "score_university", "total_score"]
titles = ["AONB Proximity", "Sea Proximity", "Low Vape/Kebab",
          "Train Station", "Far from Uni", "TOTAL SCORE"]

for ax, col, title in zip(axes.flat, plot_cols, titles):
    qualified[col].hist(ax=ax, bins=50, color="steelblue", edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")

fig.suptitle("Score Distributions (qualifying LSOAs only)", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSummary statistics (qualifying LSOAs):")
qualified[score_cols + ["total_score"]].describe().round(3)

### 21 — Top 50 results with details

In [ ]:
top50 = qualified.head(50).copy()

# Add constituency name from the lookup if available
pcon_name_cols = [c for c in lsoa_pcon.columns if "PCON" in c.upper() and "NM" in c.upper()]
if pcon_name_cols:
    LOOKUP_PCON_NM = pcon_name_cols[0]
    lsoa_to_pcon_name = dict(zip(lsoa_pcon[LOOKUP_LSOA_CD], lsoa_pcon[LOOKUP_PCON_NM]))
    top50["constituency"] = top50["lsoa_code"].map(lsoa_to_pcon_name)
    display_cols = ["rank", "lsoa_name", "constituency", "total_score"] + score_cols
else:
    display_cols = ["rank", "lsoa_name", "total_score"] + score_cols

# Set pandas display options for a nice table
with pd.option_context("display.max_rows", 50, "display.max_columns", 12,
                       "display.width", 200, "display.float_format", "{:.2f}".format):
    display(top50[display_cols])

print(f"\nBest neighbourhood: {top50.iloc[0]['lsoa_name']} "
      f"(score: {top50.iloc[0]['total_score']:.2f})")